In [1]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

spark = SparkSession.builder \
    .appName("Tugas12") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

Picked up JAVA_TOOL_OPTIONS: -XX:+IgnoreUnrecognizedVMOptions -XX:-UseBASE64Intrinsics
Picked up JAVA_TOOL_OPTIONS: -XX:+IgnoreUnrecognizedVMOptions -XX:-UseBASE64Intrinsics


26/11/19 06:10:26 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/19 06:10:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/19 06:10:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)



+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows



## A. Membangun ML Pipeline

In [3]:
# StringIndexer untuk tiga kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# OneHotEncoder untuk ketiganya
ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

# VectorAssembler: one-hot + qty + waktu_tunggu_menit
assembler = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# Algoritma klasifikasi
lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

fe_stages = [idx_menu, idx_cabang, idx_waktu, ohe, assembler]
pipeline_lr = Pipeline(stages=fe_stages + [lr])
print("Pipeline Logistic Regression terdefinisi (5 tahap feature engineering + 1 tahap model).")

Pipeline Logistic Regression terdefinisi (5 tahap feature engineering + 1 tahap model).


## B. Pelatihan & Evaluasi

In [4]:
train, test = df.randomSplit([0.8, 0.2], seed=77)
print(f"Data training: {train.count()} | Data testing: {test.count()}")

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print("=== Evaluasi Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix:")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Data training: 2411 | Data testing: 589


=== Evaluasi Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix:


+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



## C. Model Pembanding: Random Forest

In [5]:
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=42)
pipeline_rf = Pipeline(stages=fe_stages + [rf])   # lima tahap awal identik, hanya tahap terakhir diganti

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print(f"{'Model':<22}{'Accuracy':>10}{'F1':>10}{'AUC':>10}")
print(f"{'Logistic Regression':<22}{acc_lr:>10.4f}{f1_lr:>10.4f}{auc_lr:>10.4f}")
print(f"{'Random Forest':<22}{acc_rf:>10.4f}{f1_rf:>10.4f}{auc_rf:>10.4f}")

print("\nConfusion Matrix (Random Forest):")
pred_rf.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Model                   Accuracy        F1       AUC
Logistic Regression       0.7929    0.7926    0.8650
Random Forest             0.7963    0.7958    0.8474

Confusion Matrix (Random Forest):


+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  261|
|             0|       1.0|   53|
|             1|       0.0|   67|
|             1|       1.0|  208|
+--------------+----------+-----+



**Perbandingan:** Logistic Regression mencatat accuracy 0,7929, F1 0,7926, dan AUC 0,8650, sedangkan Random Forest mencatat accuracy 0,7963, F1 0,7958, dan AUC 0,8474. Random Forest unggul tipis di accuracy dan F1, tetapi selisihnya hanya ±0,003 (setara sekitar 2 dari 589 prediksi), sehingga praktis tidak berbeda. Logistic Regression justru unggul lebih jelas di AUC (selisih ±0,018), yang mengukur kemampuan memisahkan kedua kelas di semua ambang batas. Model yang lebih kompleks tidak memberi keuntungan di sini, kemungkinan karena hubungan antara waktu tunggu dan peluang ulasan positif cukup mulus dan satu arah, sehingga model sederhana sudah memadai. Berdasarkan AUC, **Logistic Regression** dipilih sebagai model terbaik.

## D. Analisis Kesalahan Model

In [6]:
# Pilih model terbaik berdasarkan AUC tertinggi
if auc_rf > auc_lr:
    nama_terbaik, pred_terbaik = "Random Forest", pred_rf
else:
    nama_terbaik, pred_terbaik = "Logistic Regression", pred_lr
print("Model terbaik (AUC tertinggi):", nama_terbaik)

salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
print(f"Jumlah prediksi salah: {salah.count()} dari {pred_terbaik.count()}")

# 10 baris yang salah diprediksi
salah.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit",
             "ulasan_positif", "prediction").show(10)

Model terbaik (AUC tertinggi): Logistic Regression


Jumlah prediksi salah: 122 dari 589


+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|  cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|      Malam|  1|                15|             1|       0.0|
|Americano|Magelang|      Siang|  2|                22|             1|       0.0|
|Americano|Magelang|       Sore|  1|                 7|             0|       1.0|
|Americano|Magelang|       Sore|  2|                11|             0|       1.0|
|Americano|Magelang|       Sore|  3|                 5|             0|       1.0|
|Americano|Magelang|       Sore|  3|                13|             1|       0.0|
|Americano|Magelang|       Sore|  5|                 6|             0|       1.0|
|Americano|Magelang|       Sore|  5|                15|             1|       0.0|
|Americano|Semarang|      Malam|  1|                 2|             0|       1.0|
|Americano|Semar

In [7]:
# Pola kesalahan: error rate per waktu tunggu
total_per_menit = pred_terbaik.groupBy("waktu_tunggu_menit").count().withColumnRenamed("count", "total")
salah_per_menit = salah.groupBy("waktu_tunggu_menit").count().withColumnRenamed("count", "salah")

total_per_menit.join(salah_per_menit, "waktu_tunggu_menit", "left").na.fill(0) \
    .withColumn("error_rate", (col("salah") / col("total")).cast("decimal(5,2)")) \
    .orderBy("waktu_tunggu_menit").show(30)

+------------------+-----+-----+----------+
|waktu_tunggu_menit|total|salah|error_rate|
+------------------+-----+-----+----------+
|                 2|   25|    1|      0.04|
|                 3|   21|    3|      0.14|
|                 4|   35|    2|      0.06|
|                 5|   22|    2|      0.09|
|                 6|   31|    6|      0.19|
|                 7|   26|    5|      0.19|
|                 8|   28|    7|      0.25|
|                 9|   28|    6|      0.21|
|                10|   23|    8|      0.35|
|                11|   19|   11|      0.58|
|                12|   18|   10|      0.56|
|                13|   29|   10|      0.34|
|                14|   26|   13|      0.50|
|                15|   19|    6|      0.32|
|                16|   28|    8|      0.29|
|                17|   22|    6|      0.27|
|                18|   19|    2|      0.11|
|                19|   26|    4|      0.15|
|                20|   18|    2|      0.11|
|                21|   24|    1|

**Pola kesalahan:** Dari 589 data uji, 122 prediksi salah (±20,7%). Kesalahan **tidak tersebar merata** menurut waktu tunggu: error rate hanya ±10% pada pesanan dengan waktu tunggu ≤ 6 menit dan ±9,5% pada ≥ 18 menit, tetapi melonjak menjadi ±45% (52 dari 115 pesanan) pada rentang 10–14 menit, dengan puncak >55% di 11–12 menit. Rentang 10–14 menit hanya mencakup ±19,5% data uji, namun menyumbang ±43% dari seluruh kesalahan. Ini sesuai dugaan: di sekitar ambang tersebut peluang ulasan positif dan negatif hampir sama, sehingga hasilnya nyaris acak dan sulit ditebak oleh model mana pun. Kesalahan di luar zona itu (mis. waktu tunggu 2 menit tetapi ulasan negatif) adalah noise bawaan data, bukan kelemahan model. Catatan: 10 baris yang ditampilkan didominasi Americano karena urutan data, bukan karena Americano bermasalah. Karena itu pola dinilai dari agregasi error rate per waktu tunggu, bukan dari 10 contoh tersebut.

### Fitur paling berpengaruh (pendukung bagian E)

In [8]:
attrs = pred_rf.schema["features"].metadata["ml_attr"]["attrs"]
fitur = sorted([a for grp in attrs.values() for a in grp], key=lambda a: a["idx"])
names = [a["name"] for a in fitur]
imp = model_rf.stages[-1].featureImportances.toArray()

print("Top 5 fitur (Random Forest):")
for nm, v in sorted(zip(names, imp), key=lambda x: -x[1])[:5]:
    print(f"{nm:<25}{v:.4f}")

Top 5 fitur (Random Forest):
waktu_tunggu_menit       0.9430
qty                      0.0113
menu_vec_Kopi Susu       0.0094
waktu_vec_Malam          0.0059
cabang_vec_Yogyakarta    0.0058


## E. Kesimpulan Bisnis
Model terbaik adalah **Logistic Regression** (AUC 0,865; accuracy 0,793; F1 0,793), sedikit di atas Random Forest (AUC 0,847). Analisis kesalahan menunjukkan model paling sering keliru pada pesanan dengan waktu tunggu **10–14 menit** (error rate ±45%, dibanding ±15% di luar rentang itu), yaitu zona abu-abu di mana peluang ulasan positif dan negatif hampir sama. Fitur yang paling berpengaruh adalah `waktu_tunggu_menit` (feature importance ±0,94 pada Random Forest), sedangkan menu, cabang, waktu pesan, dan qty hampir tidak berpengaruh.

**Rekomendasi untuk manajemen Kopi Nusantara:** (1) jadikan waktu tunggu sebagai KPI utama dengan target di bawah ±10 menit di semua cabang; (2) beri peringatan atau prioritas pada pesanan yang antreannya mendekati 10 menit sebelum memasuki zona risiko; (3) tidak perlu merombak menu atau cabang tertentu karena model tidak menemukan perbedaan antar-menu maupun antar-cabang.

**Catatan:** model hanya menunjukkan asosiasi, bukan sebab-akibat, dan dataset ini hasil simulasi sehingga temuan perlu divalidasi dengan data nyata.

In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
